# 📦 Dataset Preparation: BanglaVision40k → dataset40k

This notebook consolidates images and captions from multiple sources (BanglaVision40k subdirectories and the V5 Beyond Caption Dataset) into a unified `dataset40k` folder with sequential numeric filenames.

**Pipeline:**
1. Load captions from BanglaVision40k CSV
2. Load V5 Beyond Caption Dataset annotations
3. Copy images from source → `dataset40k/images/` with sequential IDs
4. Write `caption.txt` with `filename caption` pairs
5. Save `filename_mapping.json` for traceability


## 📥 Imports & Configuration


In [1]:
import os
import csv
import json
import shutil
from collections import defaultdict
from tqdm import tqdm


### Path Configuration

Set up source and output directories. Update `PROJECT_ROOT` to match your environment.


In [2]:
# 🔧 Root directory containing BanglaVision40k
# Auto-detect project root
_cwd = os.getcwd()
for _ in range(4):
    if os.path.isdir(os.path.join(_cwd, "BanglaVision40k")):
        break
    _p = os.path.dirname(_cwd)
    if _p == _cwd:
        break
    _cwd = _p
PROJECT_ROOT = _cwd

# 📂 Output directory for consolidated dataset
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "EXP_siglip2_Att_GRU_Ex", "dataset40k")
IMAGES_DIR = os.path.join(OUTPUT_DIR, "images")
CAPTION_FILE = os.path.join(OUTPUT_DIR, "caption.txt")
MAPPING_FILE = os.path.join(OUTPUT_DIR, "filename_mapping.json")

# 📂 Source directories
BANGLAVISION_DIR = os.path.join(PROJECT_ROOT, "BanglaVision40k")
BANGLAVISION_CSV = os.path.join(BANGLAVISION_DIR, "captions.csv")
SUBDIRS = ["BanglaView", "BNature", "Bornon"]


## 📖 Load BanglaVision40k Captions

Parse the BanglaVision40k CSV file mapping image paths to their captions.


In [3]:
def load_banglavision_csv(csv_path):
    """Load BanglaVision40k captions from CSV."""
    image_to_captions = defaultdict(list)
    with open(csv_path, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            img_path = row["image_path"].strip()
            caption = row["caption"].strip()
            image_to_captions[img_path].append(caption)
    return image_to_captions


## 📖 Load V5 Beyond Caption Dataset

Load the V5 dataset annotations which contain 5 short captions per image.


In [4]:
V5_DIR = os.path.join(BANGLAVISION_DIR, "V5 Beyond Caption Dataset")
V5_ANNOTATIONS = os.path.join(V5_DIR, "annotation.csv")
V5_IMAGES_DIR = os.path.join(V5_DIR, "Images1")


def load_v5_annotations():
    """Load V5 annotations, keeping only images with exactly 5 captions."""
    image_to_captions = {}
    with open(V5_ANNOTATIONS, "r", encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        temp = defaultdict(list)
        for row in reader:
            img = row["image"].strip()
            cap = row["short_caption_bn"].strip()
            temp[img].append(cap)
    for img, caps in temp.items():
        if len(caps) == 5:
            image_to_captions[img] = caps
    return image_to_captions


## 🔄 Load Existing Mapping (for resuming)

If the dataset was partially created before, load the existing mapping to avoid re-copying.


In [5]:
def load_existing_mapping():
    """Load previously saved filename mapping, if it exists."""
    if not os.path.exists(MAPPING_FILE):
        return {}, set()
    with open(MAPPING_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)
    new_to_old = data.get("new_to_old", {})
    copied_sources = set(os.path.abspath(p) for p in new_to_old.values())
    return new_to_old, copied_sources


## 🚀 Main Consolidation Function

The main function orchestrates the entire dataset preparation pipeline.


In [6]:
def main():
    """Consolidate BanglaVision40k + V5 into dataset40k."""
    os.makedirs(IMAGES_DIR, exist_ok=True)

    # Check for existing dataset
    existing_mapping, copied_sources = load_existing_mapping()
    if existing_mapping:
        print(f"WARNING: dataset40k already exists ({len(existing_mapping)} images)!")
        print("Running again will REGENERATE caption.txt and filename_mapping.json,")
        print("which will UNDO all cleaning (translations, garbage removal, etc.).")
        response = input("Continue? (yes/no): ").strip().lower()
        if response != "yes":
            print("Aborted.")
            return

    start_id = 1
    if existing_mapping:
        existing_ids = [int(f.replace(".jpg", "")) for f in existing_mapping if f.endswith(".jpg")]
        start_id = max(existing_ids) + 1 if existing_ids else 1

    # Load BanglaVision captions
    print("Loading BanglaVision40k captions...")
    banglavision_captions = load_banglavision_csv(BANGLAVISION_CSV)
    print(f"  {len(banglavision_captions)} unique image keys loaded")
    all_image_keys = set(banglavision_captions.keys())

    new_to_old = dict(existing_mapping)
    counter = iter(range(start_id, 10000000))
    skipped = 0

    # Copy images from each subdirectory
    print("\nCopying images...")
    for subdir in SUBDIRS:
        src_dir = os.path.join(BANGLAVISION_DIR, subdir)
        if not os.path.isdir(src_dir):
            print(f"  Warning: {src_dir} not found, skipping")
            continue
        present = [f for f in os.listdir(src_dir)
                   if os.path.isfile(os.path.join(src_dir, f))]
        for fname in tqdm(present, desc=f"  {subdir}"):
            rel_path = f"{subdir}/{fname}"
            if rel_path not in all_image_keys:
                continue
            src = os.path.join(src_dir, fname)
            if os.path.abspath(src) in copied_sources:
                continue
            if not os.path.exists(src):
                skipped += 1
                continue
            new_id = next(counter)
            new_fname = f"{new_id}.jpg"
            shutil.copy2(src, os.path.join(IMAGES_DIR, new_fname))
            new_to_old[new_fname] = src

    # Copy V5 Beyond Caption images
    print("\nProcessing V5 Beyond Caption Dataset...")
    if os.path.isdir(V5_IMAGES_DIR) and os.path.exists(V5_ANNOTATIONS):
        v5_captions = load_v5_annotations()
        print(f"  {len(v5_captions)} images with 5 short captions")
        v5_present = [
            f for f in os.listdir(V5_IMAGES_DIR)
            if f in v5_captions and os.path.isfile(os.path.join(V5_IMAGES_DIR, f))
        ]
        for fname in tqdm(v5_present, desc="  V5 Beyond Caption"):
            src = os.path.join(V5_IMAGES_DIR, fname)
            if os.path.abspath(src) in copied_sources:
                continue
            new_id = next(counter)
            new_fname = f"{new_id}.jpg"
            shutil.copy2(src, os.path.join(IMAGES_DIR, new_fname))
            new_to_old[new_fname] = src

    total_copied = len(new_to_old)
    print(f"\nTotal images in dataset: {total_copied}")
    if skipped:
        print(f"  ({skipped} files skipped due to missing source)")

    # Build old→new mapping
    old_to_new = {}
    for new_fname, src_path in new_to_old.items():
        old_fname = os.path.basename(src_path)
        old_to_new[old_fname] = new_fname

    # Write caption.txt
    print("\nWriting caption.txt...")
    v5_captions = load_v5_annotations() if os.path.exists(V5_ANNOTATIONS) else {}
    caption_count = 0
    with open(CAPTION_FILE, "w", encoding="utf-8") as out_f:
        for new_fname, src_path in tqdm(new_to_old.items(), desc="  writing captions"):
            old_fname = os.path.basename(src_path)
            parent_dir = os.path.basename(os.path.dirname(src_path))
            rel_path = f"{parent_dir}/{old_fname}"
            caps = banglavision_captions.get(rel_path, [])
            if not caps:
                caps = v5_captions.get(old_fname, [])
            if caps:
                for cap in caps:
                    out_f.write(f"{new_fname} {cap}\n")
                    caption_count += 1

    print(f"  {caption_count} total caption lines written")

    # Save filename mapping
    with open(MAPPING_FILE, "w", encoding="utf-8") as f:
        json.dump({"old_to_new": old_to_new, "new_to_old": new_to_old},
                  f, ensure_ascii=False, indent=2)
    print(f"Filename mapping saved to {MAPPING_FILE}")

    print(f"\n{"=" * 50}")
    print(f"dataset40k consolidation complete!")
    print(f"  Images: {total_copied}")
    print(f"  Caption lines: {caption_count}")
    print(f"  Images dir: {IMAGES_DIR}")
    print(f"  Caption file: {CAPTION_FILE}")
    print(f"{"=" * 50}")


## ▶️ Run Dataset Preparation

Execute the main function to start consolidating the dataset.


In [7]:
if __name__ == "__main__":
    main()


Running again will REGENERATE caption.txt and filename_mapping.json,
which will UNDO all cleaning (translations, garbage removal, etc.).


Continue? (yes/no):  yes


Loading BanglaVision40k captions...
  44535 unique image keys loaded

Copying images...


  Bornon: 100%|██████████| 4088/4088 [00:00<00:00, 329965.26it/s]



Processing V5 Beyond Caption Dataset...
  289 images with 5 short captions


  V5 Beyond Caption: 100%|██████████| 289/289 [00:00<00:00, 296732.89it/s]



Total images in dataset: 44156

Writing caption.txt...


  writing captions: 100%|██████████| 44156/44156 [00:00<00:00, 83304.41it/s]


  218129 total caption lines written
Filename mapping saved to D:\Python Projects\A_Final_Final_Final\EXP_siglip2_Att_GRU_Ex\dataset40k\filename_mapping.json

dataset40k consolidation complete!
  Images: 44156
  Caption lines: 218129
  Images dir: D:\Python Projects\A_Final_Final_Final\EXP_siglip2_Att_GRU_Ex\dataset40k\images
  Caption file: D:\Python Projects\A_Final_Final_Final\EXP_siglip2_Att_GRU_Ex\dataset40k\caption.txt
